# **Data Cleaning**

## Objectives

* Correct all data errors via type correction, removal, or replacement as required

## Inputs

* outputs/datasets/collection/HotelBookings.csv

## Outputs

* Generate cleaned dataset saved as outputs/datasets/cleaned/HotelBookingsClean.csv

## Decisions from Revenue Manager

* Duplicate bookings should remain in place
* Bookings with no guests (adults, children or babies) should be removed
* Values > 4 for babies or children and > 2 for car park spaces should be replaced with their column's median value
* Bookings with >= 5 adults should be removed
* Isolated high and low outliers for adr should be removed
* Missing data in children should be replaced by 0
* Missing data in agent and company should be replaced by 0
* Missing data in country should be replaced my the mode value 

## Standard data cleaning

* Drop variables: `['reservation_status', 'reservation_status_date']`
* Perform dtype correction on children (float -> int)
* Ensure all numeric-as-categorical features are converted to 'category'


---

## Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/home/niall/PP4/cancel-protect/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir


'/home/niall/PP4/cancel-protect'

## Load Data

In [3]:
import pandas as pd
df = pd.read_csv("outputs/datasets/collection/HotelBookings.csv")
df.head(3)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


### Missingness

In [4]:
# The following function is taken from the 'Chrunometer' walkthrough
def EvaluateMissingData(df):
    missing_data_absolute = df.isnull().sum()
    missing_data_percentage = round(missing_data_absolute/len(df)*100, 2)
    df_missing_data = (pd.DataFrame(
                            data={"RowsWithMissingData": missing_data_absolute,
                                   "PercentageOfDataset": missing_data_percentage,
                                   "DataType": df.dtypes}
                                    )
                          .sort_values(by=['PercentageOfDataset'], ascending=False)
                          .query("RowsWithMissingData > 0")
                          )

    return df_missing_data

In [5]:
EvaluateMissingData(df)

,RowsWithMissingData,PercentageOfDataset,DataType
company,112593,94.31,float64
agent,16340,13.69,float64
country,488,0.41,object
children,4,0.00,float64


* As per received guidance [Revenue Manager's Notebook](/jupyter_notebooks/03_rm_analysis.ipynb), replace missing values in `children`, `agent` and `company` with '0'

In [6]:
def replace_zeros(df):
    cols_to_replace_zero = ["children", "company", "agent"]
    for col in cols_to_replace_zero:
        df[col] = df[col].fillna(0)

replace_zeros(df)

In [7]:
df.head(10)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,0.0,0.0,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,0.0,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,0.0,0,Transient,98.0,0,1,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,0.0,0,Transient,98.0,0,1,Check-Out,2015-07-03
6,Resort Hotel,0,0,2015,July,27,1,0,2,2,...,No Deposit,0.0,0.0,0,Transient,107.0,0,0,Check-Out,2015-07-03
7,Resort Hotel,0,9,2015,July,27,1,0,2,2,...,No Deposit,303.0,0.0,0,Transient,103.0,0,1,Check-Out,2015-07-03
8,Resort Hotel,1,85,2015,July,27,1,0,3,2,...,No Deposit,240.0,0.0,0,Transient,82.0,0,1,Canceled,2015-05-06
9,Resort Hotel,1,75,2015,July,27,1,0,3,2,...,No Deposit,15.0,0.0,0,Transient,105.5,0,0,Canceled,2015-04-22


* Re-evaluate missingness

In [8]:
EvaluateMissingData(df)

,RowsWithMissingData,PercentageOfDataset,DataType
country,488,0.41,object


* For `country` we should replace the missing values with the variable mode value

In [9]:
df["country"].mode()

0    PRT
Name: country, dtype: object

In [10]:
df["country"] =df["country"].fillna("PRT")
EvaluateMissingData(df)

,RowsWithMissingData,PercentageOfDataset,DataType


---

## Outliers

1. Bookings with no guests of any description should be removed

In [11]:
df.shape

(119390, 32)

In [12]:
no_guest_Df = df[((df["adults"] == 0) & (df["children"] == 0) & (df["babies"] == 0))]
no_guest_Df.shape

(180, 32)

In [13]:
df = df.drop(no_guest_Df.index)
df.shape

(119210, 32)

2. Bookings with 5 or more adults should be removed

In [14]:
many_adults_df = df[df["adults"] >= 5]
many_adults_df.shape

(16, 32)

In [15]:
df = df.drop(many_adults_df.index)
df.shape

(119194, 32)

3. Bookings with more than 4 babies or children and bookings with more than 2 car park spaces should have the values replaced with the column's median value

In [16]:
high_values_df = df[((df["children"] > 4) | (df["babies"] > 4) | (df["required_car_parking_spaces"] > 2))]
high_values_df               

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
328,Resort Hotel,1,55,2015,July,29,12,4,10,2,...,No Deposit,8.0,0.0,0,Contract,133.16,0,1,No-Show,2015-07-12
29045,Resort Hotel,0,26,2017,March,11,14,0,5,2,...,No Deposit,0.0,0.0,0,Transient-Party,40.00,8,1,Check-Out,2017-03-19
29046,Resort Hotel,0,138,2017,March,12,19,2,2,2,...,No Deposit,0.0,0.0,122,Transient-Party,80.00,8,0,Check-Out,2017-03-23
38117,Resort Hotel,0,205,2017,June,26,26,3,10,2,...,No Deposit,250.0,0.0,0,Transient,111.00,3,0,Check-Out,2017-07-09
46619,City Hotel,0,37,2016,January,3,12,0,2,2,...,No Deposit,9.0,0.0,0,Transient,84.45,0,1,Check-Out,2016-01-14
78656,City Hotel,0,11,2015,October,42,11,2,1,1,...,No Deposit,95.0,0.0,0,Transient-Party,95.00,0,0,Check-Out,2015-10-14
102762,City Hotel,0,13,2016,December,50,5,1,0,1,...,No Deposit,9.0,0.0,0,Transient,96.00,3,0,Check-Out,2016-12-06
110812,City Hotel,0,30,2017,April,17,29,2,1,2,...,No Deposit,9.0,0.0,0,Transient-Party,153.33,3,2,Check-Out,2017-05-02


* Replace all values above their threshold value with the column median value

In [17]:
def replace_median(df):
    column_thresholds = {"children": 4, "babies": 4, "required_car_parking_spaces":2}
    for col, threshold in column_thresholds.items():
        median_value = df[col].median()
        df.loc[df[col] > threshold, col] = median_value
    return df

df = replace_median(df)


In [18]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,0.0,0.0,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,0.0,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,0.0,0,Transient,98.0,0,1,Check-Out,2015-07-03


* Re-run high values check to ensure changes have taken effect

In [20]:
high_values_df = df[((df["children"] > 4) | (df["babies"] > 4) | (df["required_car_parking_spaces"] > 2))]
high_values_df

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date


4. Remove high and low isolated extreme values for `adr`

In [22]:
df["adr"].describe()

count    119194.000000
mean        101.982780
std          50.423552
min          -6.380000
25%          69.522500
50%          94.970000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

In [23]:
adr_extremes = df[((df["adr"] > 600) | (df["adr"] < 0))]
adr_extremes

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
14969,Resort Hotel,0,195,2017,March,10,5,4,6,2,...,No Deposit,273.0,0.0,0,Transient-Party,-6.38,0,0,Check-Out,2017-03-15
48515,City Hotel,1,35,2016,March,13,25,0,1,2,...,Non Refund,12.0,0.0,0,Transient,5400.00,0,0,Canceled,2016-02-19


In [24]:
df = df.drop(adr_extremes.index)
df["adr"].describe()

count    119192.000000
mean        101.939239
std          48.031041
min           0.000000
25%          69.527500
50%          94.970000
75%         126.000000
max         510.000000
Name: adr, dtype: float64

---

## Type Conversions

1. Children float64 should be int64 since partial children not possible

In [27]:
df["children"].dtypes

dtype('float64')

In [28]:
df["children"] = df["children"].astype("int64")
df["children"].dtypes

dtype('int64')

2. Numeric-as-categoric conversion

In [29]:
df.dtypes

hotel                              object
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                 object
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                            int64
babies                              int64
meal                               object
country                            object
market_segment                     object
distribution_channel               object
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                 object
assigned_room_type                 object
booking_changes                     int64
deposit_type                       object
agent                             

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In case you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create here your folder
  # os.makedirs(name='')
except Exception as e:
  print(e)
